In [ ]:
import time
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM, AutoModelForCausalLM

# TRANSFORMER IN torch.nn

Gli stessi blocchi esistono anche in PyTorch, ma **senza pesi addestrati** e senza embedding: lavorano su vettori di forma (batch, token, d_model), quindi Word Embedding, Positional Embedding e il `Linear` finale vanno aggiunti a mano.

## Transformer

![Transformer](imgs/transformer.png)

https://poloclub.github.io/transformer-explainer/

In [ ]:
from torch import nn

D_MODEL, N_HEADS, N_LAYERS, D_FF = 32, 4, 2, 64      # dimensione dei vettori, teste di attenzione, blocchi, rete feed-forward
B, S, T = 2, 7, 5                                    # batch, token della sorgente, token della destinazione
src = torch.randn(B, S, D_MODEL)                     # al posto di Word + Positional Embedding della frase sorgente
tgt = torch.randn(B, T, D_MODEL)                     # ... e della frase di destinazione

# Maschera di padding: in PyTorch True = token da ignorare (al contrario di attention_mask di Hugging Face, dove 0 = padding)
src_pad = torch.zeros(B, S, dtype=torch.bool)
src_pad[1, :2] = True                                # la seconda frase ha 2 token di padding all'inizio
# Maschera causale: -inf sopra la diagonale, così ogni token guarda solo se stesso e quelli precedenti
causal = nn.Transformer.generate_square_subsequent_mask(T)
print(f"Maschera causale:\n{causal}\n")

def n_params(module):
    return sum(p.numel() for p in module.parameters())

# ENCODER ONLY (come DistilBERT): blocchi di self-attention + feed-forward, ogni token guarda tutti gli altri
encoder_only = nn.TransformerEncoder(
    nn.TransformerEncoderLayer(D_MODEL, N_HEADS, dim_feedforward=D_FF, batch_first=True), num_layers=N_LAYERS)
memory = encoder_only(src, src_key_padding_mask=src_pad)
print(f"nn.TransformerEncoder:  {tuple(src.shape)} -> {tuple(memory.shape)}   ({n_params(encoder_only)} parametri)")

# DECODER DI UN ENCODER-DECODER (come in MarianMT): self-attention causale + cross-attention sugli stati dell'encoder
decoder = nn.TransformerDecoder(
    nn.TransformerDecoderLayer(D_MODEL, N_HEADS, dim_feedforward=D_FF, batch_first=True), num_layers=N_LAYERS)
out = decoder(tgt, memory, tgt_mask=causal, memory_key_padding_mask=src_pad)
print(f"nn.TransformerDecoder:  {tuple(tgt.shape)} + memoria {tuple(memory.shape)} -> {tuple(out.shape)}   ({n_params(decoder)} parametri)")

# ENCODER-DECODER COMPLETO: nn.Transformer contiene un TransformerEncoder e un TransformerDecoder
transformer = nn.Transformer(D_MODEL, N_HEADS, num_encoder_layers=N_LAYERS, num_decoder_layers=N_LAYERS,
                             dim_feedforward=D_FF, batch_first=True)
out = transformer(src, tgt, tgt_mask=causal, src_key_padding_mask=src_pad, memory_key_padding_mask=src_pad)
print(f"nn.Transformer:         {tuple(src.shape)} e {tuple(tgt.shape)} -> {tuple(out.shape)}   ({n_params(transformer)} parametri)")

# DECODER ONLY (come GPT-2): nn.TransformerDecoder non va bene, perché si aspetta sempre gli stati di un encoder.
# Si usa un TransformerEncoder con la maschera causale: blocchi di sola self-attention in cui nessuno guarda avanti
decoder_only = nn.TransformerEncoder(
    nn.TransformerEncoderLayer(D_MODEL, N_HEADS, dim_feedforward=D_FF, batch_first=True), num_layers=N_LAYERS)
out = decoder_only(tgt, mask=causal)
print(f"decoder only:           {tuple(tgt.shape)} -> {tuple(out.shape)}   ({n_params(decoder_only)} parametri)")

# MODELLI PRE-ADDESTRATI

Tre Transformer già addestrati, scaricati da Hugging Face con la libreria `transformers`
(`pip install transformers sentencepiece`; al primo uso scaricano circa 1 GB di pesi).

| | Encoder only | Encoder-decoder | Decoder only |
|---|---|---|---|
| Modello | DistilBERT | MarianMT | GPT-2 |
| Ogni token guarda | tutti gli altri | encoder: tutti; decoder: i precedenti e l'encoder | solo i precedenti |
| Compito | classificazione | traduzione | generazione |
| Nei nostri notebook | 05 e 09 (IMDB) | 07 e 08 (Tatoeba) | 06 (Promessi Sposi) |

In [ ]:
DEVICE = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device('cpu')
print(f"DEVICE: {DEVICE}")

def count_parameters(model):
    return f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f} milioni di parametri"

# ENCODER ONLY: DistilBERT

DistilBERT è una versione ridotta di BERT, già addestrata a classificare frasi di recensioni di film in inglese (SST-2) come positive o negative: lo stesso compito di IMDB. Ogni token guarda tutti gli altri, a sinistra e a destra, e la classificazione parte dal vettore del token speciale `[CLS]`.

In [ ]:
ENCODER_NAME = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
enc_tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
encoder = AutoModelForSequenceClassification.from_pretrained(ENCODER_NAME).to(DEVICE).eval()
print(f"{ENCODER_NAME}: {count_parameters(encoder)}\n")

# TOKENIZZAZIONE: sotto-parole (## = continuazione di una parola) e token speciali [CLS] all'inizio e [SEP] alla fine
review = "This movie was a complete waste of time, but the soundtrack is unforgettable."
print(enc_tokenizer.convert_ids_to_tokens(enc_tokenizer(review)['input_ids']))

In [ ]:
reviews = ["This movie was a complete waste of time.",
           "One of the best films I have ever seen, the actors are wonderful!",
           "The plot is predictable, but the soundtrack is unforgettable."]

# Padding a destra fino alla frase più lunga; attention_mask vale 0 sul padding, che così non riceve attenzione
inputs = enc_tokenizer(reviews, padding=True, truncation=True, return_tensors='pt').to(DEVICE)
print(f"attention_mask:\n{inputs['attention_mask']}\n")

with torch.no_grad():
    outputs = encoder(**inputs, output_hidden_states=True)
print(f"Uscita dell'encoder: {tuple(outputs.hidden_states[-1].shape)} = (frasi, token, dimensione)")
print(f"Logit: {tuple(outputs.logits.shape)} = (frasi, classi)\n")

probs = torch.softmax(outputs.logits, dim=-1)
for review, p in zip(reviews, probs):
    print(f"{encoder.config.id2label[p.argmax().item()]:<9} {p.max():.3f}   {review}")

# ENCODER-DECODER: MarianMT

MarianMT è un Transformer encoder-decoder addestrato a tradurre dall'italiano all'inglese, lo stesso compito dei notebook 07 e 08: l'encoder legge la frase italiana e il decoder genera l'inglese guardando l'encoder con l'attenzione. È stato addestrato sul corpus OPUS, che comprende anche le frasi di Tatoeba, quindi non va confrontato con i nostri modelli sul loro test set.

In [ ]:
SEQ2SEQ_NAME = "Helsinki-NLP/opus-mt-it-en"
s2s_tokenizer = AutoTokenizer.from_pretrained(SEQ2SEQ_NAME)
seq2seq = AutoModelForSeq2SeqLM.from_pretrained(SEQ2SEQ_NAME).to(DEVICE).eval()
print(f"{SEQ2SEQ_NAME}: {count_parameters(seq2seq)}\n")

sentences = ["Tom è molto stanco.", "Dov'è la stazione?", "Non ho fame.", "Amo andare al parcuo!"]
inputs = s2s_tokenizer(sentences, padding=True, return_tensors='pt').to(DEVICE)
with torch.no_grad():
    generated = seq2seq.generate(**inputs, num_beams=1, max_new_tokens=40)   # num_beams=1: greedy, come nei nostri notebook
for sentence, translation in zip(sentences, s2s_tokenizer.batch_decode(generated, skip_special_tokens=True)):
    print(f"{sentence}\n  -> {translation}")

In [ ]:
def translate(sentence, delay=0.1):
    """Come translate del notebook 07: l'encoder legge la frase una volta, il decoder genera un token alla volta."""
    inputs = s2s_tokenizer(sentence, return_tensors='pt').to(DEVICE)
    print(s2s_tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))     # sotto-parole SentencePiece: ▁ = inizio di parola
    print("  -> ", end='')
    with torch.no_grad():
        encoder_outputs = seq2seq.get_encoder()(**inputs)                   # gli stati dell'encoder, calcolati una volta sola
        decoder_ids = torch.tensor([[seq2seq.generation_config.decoder_start_token_id]], device=DEVICE)   # come <sos>
        printed = ''
        for _ in range(40):
            # Il decoder Transformer non ha uno stato come la GRU: a ogni passo rilegge tutti i token già generati
            # (generate tiene in memoria i calcoli già fatti per non ripeterli)
            logits = seq2seq(encoder_outputs=encoder_outputs, attention_mask=inputs['attention_mask'],
                             decoder_input_ids=decoder_ids).logits         # (1, token generati, vocabolario)
            next_id = logits[0, -1].argmax().item()                        # il token più probabile
            if next_id == s2s_tokenizer.eos_token_id:                       # </s>: come <eos>
                break
            decoder_ids = torch.cat([decoder_ids, torch.tensor([[next_id]], device=DEVICE)], dim=1)
            text = s2s_tokenizer.decode(decoder_ids[0], skip_special_tokens=True)
            print(text[len(printed):], end='', flush=True)                 # solo la parte nuova
            printed = text
            time.sleep(delay)
    print("\n")

for sentence in sentences:
    translate(sentence)

# DECODER ONLY: GPT-2

GPT-2 small riaddestrato sull'italiano (GroNLP). Fa la stessa cosa del notebook 06: predice il token successivo e lo aggiunge al testo, ma ha più di 100 milioni di parametri ed è stato addestrato su molti testi. Ogni token guarda solo quelli precedenti (attenzione causale) e il contesto può arrivare a 1024 token, invece delle ultime 20 parole.

In [ ]:
DECODER_NAME = "GroNLP/gpt2-small-italian"
dec_tokenizer = AutoTokenizer.from_pretrained(DECODER_NAME)
decoder = AutoModelForCausalLM.from_pretrained(DECODER_NAME).to(DEVICE).eval()
print(f"{DECODER_NAME}: {count_parameters(decoder)}\n")

# TOKENIZZAZIONE: sotto-parole BPE, Ġ = spazio prima della parola
print(dec_tokenizer.convert_ids_to_tokens(dec_tokenizer("Quel ramo del lago di Como, che volge a mezzogiorno")['input_ids']))

In [ ]:
def generate(prompt, n_tokens=60, temperature=0.8, delay=0.05):
    """Come generate del notebook 06: a ogni passo il modello predice il token successivo e lo aggiunge al testo."""
    ids = dec_tokenizer(prompt, return_tensors='pt')['input_ids'].to(DEVICE)
    printed = dec_tokenizer.decode(ids[0])
    print(printed, end='')
    with torch.no_grad():
        for _ in range(n_tokens):
            logits = decoder(input_ids=ids).logits[0, -1]                  # logit del token successivo
            probs = torch.softmax(logits / temperature, dim=-1)            # temperatura come nel notebook 06
            next_id = torch.multinomial(probs, 1)                          # un token estratto secondo le probabilità
            ids = torch.cat([ids, next_id.unsqueeze(0)], dim=1)            # il token generato entra nel contesto
            text = dec_tokenizer.decode(ids[0])
            print(text[len(printed):], end='', flush=True)                 # solo la parte nuova
            printed = text
            time.sleep(delay)
    print("\n")

torch.manual_seed(0)
generate("Quel ramo del lago di Como")
generate("Renzo e Lucia")

In [ ]:
# Lo stesso con generate di transformers, che offre anche altre strategie (top-k, beam search, ...)
inputs = dec_tokenizer("Renzo e Lucia", return_tensors='pt').to(DEVICE)
with torch.no_grad():
    output = decoder.generate(**inputs, max_new_tokens=60, do_sample=True, temperature=0.8, top_k=50,
                              pad_token_id=dec_tokenizer.eos_token_id)   # GPT-2 non ha un token di padding
print(dec_tokenizer.decode(output[0], skip_special_tokens=True))